# Dataset check: Tolstoy in English translation

This notebook checks the dataset in this repository by loading `metadata.csv` into a pandas DataFrame and performs the checks required:

1. the number of items;
2. every path in the `file` column points to an existing file in `data/`, and every file in `data/` has exactly one row in `metadata.csv` (the cleaned copies in `data_cleaned/` are read separately in section 5);
3. all IDs are unique;
4. the number of missing values per column;
5. a basic summary of the dataset.

All paths are relative to the repository root, so the notebook runs on any computer with the repository and the packages from `requirements.txt`.

## Setup

In [1]:
import os
import pandas as pd

DATA_DIR = "data"
CLEANED_DIR = "data_cleaned"

metadata = pd.read_csv("metadata.csv")
metadata.head()

,id,file,source,licence,title,author,translator,work,genre,language,pg_id,release_date
0,tol_243_forged_coupon,data/tol_243_forged_coupon.txt,https://www.gutenberg.org/ebooks/243,Public domain in the US (Project Gutenberg eBo...,"The forged coupon, and other stories",Leo Tolstoy,NaN,The Forged Coupon,short stories,en,243,2006-03-15
1,tol_689_kreutzer_sonata,data/tol_689_kreutzer_sonata.txt,https://www.gutenberg.org/ebooks/689,Public domain in the US (Project Gutenberg eBo...,The Kreutzer Sonata and Other Stories,Leo Tolstoy,Benjamin Ricketson Tucker,The Kreutzer Sonata,short stories,en,689,2006-03-18
2,tol_985_father_sergius,data/tol_985_father_sergius.txt,https://www.gutenberg.org/ebooks/985,Public domain in the US (Project Gutenberg eBo...,Father Sergius,Leo Tolstoy,Aylmer Maude,Father Sergius,novella,en,985,2006-02-05
3,tol_986_master_and_man,data/tol_986_master_and_man.txt,https://www.gutenberg.org/ebooks/986,Public domain in the US (Project Gutenberg eBo...,Master and Man,Leo Tolstoy,Aylmer Maude,Master and Man,novella,en,986,2006-02-05
4,tol_1399_anna_karenina,data/tol_1399_anna_karenina.txt,https://www.gutenberg.org/ebooks/1399,Public domain in the US (Project Gutenberg eBo...,Anna Karenina,Leo Tolstoy,Constance Garnett,Anna Karenina,novel,en,1399,1998-07-01


## 1. Number of items

In [2]:
print("Number of items:", len(metadata))

Number of items: 46


## 2. Every row has a file in `data/`, and every file has a row

Only the original files are checked here, not the cleaned versions in `data_cleaned/`.

In [3]:
files_in_folder = []
for name in os.listdir(DATA_DIR):
    path = DATA_DIR + "/" + name
    if not name.startswith(".") and os.path.isfile(path):
        files_in_folder.append(path)

listed = list(metadata["file"])

for path in listed:
    assert path in files_in_folder, f"Missing file on disk: {path}"

for path in files_in_folder:
    assert listed.count(path) == 1, f"Expected count 1 in metadata.csv for: {path}"

assert len(listed) == len(files_in_folder), "Mismatch between metadata rows and folder files"

empty_files = []
for path in files_in_folder:
    if os.path.getsize(path) == 0:
        empty_files.append(path)
assert not empty_files, f"Empty files in data/: {empty_files}"

print("File check finished")


File check finished


In [4]:
files = list(metadata["file"])
ids = list(metadata["id"])

for i in range(len(files)):
    file_name = os.path.basename(files[i])
    expected_name = f"{ids[i]}.txt"
    assert file_name == expected_name, f"Expected {expected_name}, got {file_name}"

print(f"All {len(files)} file names follow '<id>.txt'")


All 46 file names follow '<id>.txt'


## 3. All IDs are unique

In [5]:
print("Repeated IDs:", metadata["id"].duplicated().sum())
assert not metadata["id"].duplicated().sum()

Repeated IDs: 0


## 4. Missing values per column


In [6]:
metadata.isna().sum()

id              0
file            0
source          0
licence         0
title           0
author          0
translator      6
work            0
genre           0
language        0
pg_id           0
release_date    0
dtype: int64

## 5. Basic summary of the dataset

### Values of the categorical columns

In [7]:
metadata["author"].value_counts()

author
Leo Tolstoy    46
Name: count, dtype: int64

### Items per translator

In [8]:
metadata["translator"].value_counts().sort_index()

translator
Aline Delano                  1
Aylmer Maude                 14
Benjamin Ricketson Tucker     1
C. J. Hogarth                 3
Constance Garnett             3
Francis Davis Millet          1
Huntington Smith              1
Isabel Florence Hapgood       5
Leo Wiener                    2
Louise Maude                  1
Nathan Haskell Dole           3
R. S. Townsend                1
Rose Strunsky                 1
V. G. Chertkov                2
William E. Smith              1
Name: count, dtype: int64

### Items per genre

In [9]:
metadata["genre"].value_counts().sort_index()

genre
children's stories    2
diary                 1
drama                 7
essay                 7
novel                 4
novella               8
short stories         8
short story           1
treatise              8
Name: count, dtype: int64

### Item sizes

In [10]:
raw_counts = []
cleaned_counts = []

for file_path in metadata["file"]:
    with open(file_path, encoding="utf-8") as f:
        raw_counts.append(len(f.read().split()))

    file_name = os.path.basename(file_path)
    cleaned_path = os.path.join(CLEANED_DIR, file_name)
    with open(cleaned_path, encoding="utf-8") as f:
        cleaned_counts.append(len(f.read().split()))

metadata["words_raw"] = raw_counts
metadata["words_cleaned"] = cleaned_counts

print(f"Total words in data/: {sum(raw_counts)}")
print(f"Total words in data_cleaned/: {sum(cleaned_counts)}")


Total words in data/: 3759239
Total words in data_cleaned/: 3618335


In [11]:
metadata.nlargest(3, "words_cleaned")[["title", "translator", "genre", "words_cleaned"]]


,title,translator,genre,words_cleaned
8,War and Peace,Aylmer Maude,novel,563286
4,Anna Karenina,Constance Garnett,novel,349763
34,The Kingdom of God is Within You; What is Art?,Aline Delano,treatise,192440


In [12]:
metadata.nsmallest(3, "words_cleaned")[["title", "translator", "genre", "words_cleaned"]]

,title,translator,genre,words_cleaned
10,The Census in Moscow,Isabel Florence Hapgood,essay,4217
24,The Cause of it All,Aylmer Maude,drama,4404
29,Where Love is There God is Also,Nathan Haskell Dole,short story,5204


## 6. Conclusion

All checks pass: the dataset has 46 items, every row points to exactly one existing file, every file in `data/` is described by exactly one row, all IDs are unique, and the only missing values are six unnamed translators, as documented. Every item also has a cleaned copy that can be read in `data_cleaned/` (section 5).
